# Step 6: Category data analysis

This notebook provides tools for analyzing the categories applied to the videos based on our category system.
The outputs are stored in the output folder.

In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.transforms as mtransforms
import matplotlib.colors as mcolors
import random
import pprint
import yaml
from glob import glob

In [ ]:
fonts = {'en': 'NotoSansJP-Light.otf', 'ja': 'NotoSansJP-Light.otf', 'ko': 'NotoSansKR-Light.otf', 'zh-Hant': 'NotoSansTC-Light.otf', 'zh-Hans': 'NotoSansSC-Light.otf'}

dataset_config = "./config/dataset_config.yml"
catfile = '*combined_cleaned_data.csv'
commentsfile = '_comments_cleaned_langdetect.csv'
metadatafile = '_metadata_cleaned_langinfo.csv'

output = './output/'

In [ ]:
# data screening

print(os.path.abspath(dataset_config))
print(os.path.getsize(dataset_config))
with open(dataset_config, "r") as f:
    config = yaml.safe_load(f)
# Expand paths relative to working dir
directories = {
    key: {
        'wd': (
            [os.path.join(os.getcwd(), path) for path in value['wd']]
            if value['wd'] != None
            else []
        ),
        'catdir': (
            os.path.join(os.getcwd(), value['catdir'])
            if value['catdir'] != None
            else ''
        ),
    }
    for key, value in config.items()
}

pprint.pprint(directories)

In [ ]:
glasbey_pastel = [
    "#e88c8f", "#ee9c79", "#efb567", "#e4c95c", "#c8d764",
    "#9cd67c", "#6fcea0", "#58c2c2", "#5ab0db", "#7395eb",
    "#927ee8", "#b46dd7", "#d062bc", "#e75f9f", "#e970b3",
    "#e687cc", "#d6a0dd", "#bfb5e8", "#a3c6e8", "#8cd6dc",
    "#7cd4be", "#84d090", "#a0cb6e", "#c2c05a", "#d3ae54",
    "#d79763", "#d87e76", "#d56b8b", "#c868a7", "#b173c0",
    "#9a85cc"
]

# Okabe–Ito palette (canonical + extended)
okabe_ito = [
    "#E69F00", "#56B4E9", "#009E73", "#F0E442",
    "#0072B2", "#D55E00", "#CC79A7", "#999999",
    "#CC3311", "#0072B2", "#009E73", "#AA4499"
]

tol20 = [
    "#4477AA", "#CC6677", "#44AA77", "#AA4499", "#DDCC77",
    "#332288", "#DDAA77", "#6699CC", "#661100", "#999933",
    "#882255", "#77DD88", "#AA4466", "#DD77AA", "#117733",
    "#77AADD", "#88CCEE", "#DDDDDD", "#44AA99", "#1F77B4",
]

In [ ]:
datadict = {}
for game in directories.keys():
    catf = glob(os.path.join(directories[game]['catdir'], catfile))
    if len(catf) < 1 or len(catf) > 1:
        print('number of category files os not 1')
    else:
        datadict[game] = pd.read_csv(catf[0])
print(datadict.keys())

In [ ]:
# simplifying chinese

for game in datadict.keys():
    print(game)
    datadict[game]['videoSearchRegion'] = datadict[game]['videoSearchRegion'].replace(['zh-hans', 'zh-hant'], 'ch')
    print(datadict[game]['Content Type (choose 0-1)'].unique())

In [ ]:
def createstatisticstables (game, data):
       
    # create statistic of terms distribution used
    selected_columns = [
        "Content match (yes no)", "Language Match (yes no)",
        "Main Video Source (choose 1)", "Formal Elements (multiple possible)",
        "Other Significant Tags (please describe new tags in the tag description column)",
        "YouTube Shorts (yes no)", "Other Noteworthy Formal Elements",
        "Content Type (choose 0-1)", "Content Focus (choose 0-2)",
        "Other Noteworthy Content Elements", "Values (Subjective Evaluation) muliple possible",
        "Other Noteworthy Evaluation", "new tags and other memos"
    ]
    group_column = 'videoId' #replace with video_status later?
    #group_column = 'video_status'

    # Explode and save each distribution to its own sheet, not per videoID and per language
    with pd.ExcelWriter(os.path.join(output, f'ytma_{game}_term_distributions_combined.xlsx'), engine='xlsxwriter') as writer:
        for col in selected_columns:
            # Explode list-like column
            exploded_df = data[['videoSearchRegion', col]].explode(col).dropna(subset=[col])
            exploded_df.columns = ['videoSearchRegion', 'term']

            # Group by language and term, count occurrences
            grouped = exploded_df.groupby(['videoSearchRegion', 'term']).size().reset_index(name='count')

            # Pivot to get per-language distribution
            pivot = grouped.pivot_table(index='term', columns='videoSearchRegion', values='count', fill_value=0)

            # Add aggregate total across all languages
            pivot['total'] = pivot.sum(axis=1)

            # Optional: sort by total
            pivot = pivot.sort_values(by='total', ascending=False)

            # Save to Excel
            pivot.to_excel(writer, sheet_name=col[:31])

    # Explode and save each distribution to its own sheet, per videoID
    with pd.ExcelWriter(os.path.join(output, f'ytma_{game}_term_distributions_exploded.xlsx'), engine='xlsxwriter') as writer:
        for col in selected_columns:
            exploded_df = data[[group_column, col]].explode(col)
            exploded_df.columns = [group_column, 'term']

            # Group and pivot
            grouped = exploded_df.groupby([group_column, 'term']).size().reset_index(name='count')
            pivot = grouped.pivot_table(index='term', columns=group_column, values='count', fill_value=0)

            # Save to sheet
            pivot.to_excel(writer, sheet_name=col[:31]) 
    return True

for g in datadict.keys():
    createstatisticstables(g, datadict[g])

In [ ]:
focus_columns = [
    "Main Video Source (choose 1)", "Formal Elements (multiple possible)",
    "Other Significant Tags (please describe new tags in the tag description column)",
    "YouTube Shorts (yes no)",
    "Content Type (choose 0-1)", "Content Focus (choose 0-2)",
    "Values (Subjective Evaluation) muliple possible"
]
exclude = ['link', 'videoId', 'new tags and other memos', 'source_file', 'content_count', 'Other Noteworthy Evaluation', 'Other Noteworthy Content Elements', 'Other Noteworthy Formal Elements']
plot_order = ["ja", "ko", "ch", "en"]

In [ ]:
def investigatefacet(df, col, facet):
    if str(facet).lower() == 'nan':
        return df[df[col].isnull()]
    else:
        return df[df[col] == facet]

# get statistics of data

In [ ]:
for game in datadict:
    print(game)
    totals = 0
    for language in plot_order:
        if language in datadict[game]['videoSearchRegion'].unique():
            print(language)
            n_language = len(datadict[game][datadict[game]['videoSearchRegion'] == language])
            print(n_language)
            totals += n_language
    print('total for game: ', totals)

# visualize data overview

compare Zelda and SF for the three categories, all video tags, all content, all values, stacked bars for four languages two games side by side.
content type and subjective eval.

In [ ]:
def plot_nested_stacked_bars(
    data,
    misc_threshold=0.0,
    misc_label="Misc",
    legend_loc="right",
    save_as_png=""
):

    # ----------------------------------------------------
    # 1. COLLECT ALL CATEGORIES AND BUILD ABSOLUTE DATA
    # ----------------------------------------------------
    abs_dict = {}
    all_cats = []

    for top_key, langs in data.items():
        df = pd.DataFrame(langs)

        df.index = (
            df.index.astype(str)
            .str.strip()
            .str.replace("nan", "Unclassified", case=False)
            .str.replace("#ref!", "Unclassified", case=False)
            .str.replace(";", "/", regex=False)
        )

        df = df.groupby(df.index).sum().fillna(0)

        if misc_threshold > 0:
            totals = df.sum(axis=1)
            global_total = totals.sum()
            misc_mask = totals / global_total < misc_threshold
            if misc_mask.any():
                misc_sum = df[misc_mask].sum()
                df = df[~misc_mask]
                df.loc[misc_label] = misc_sum

        abs_dict[top_key] = df
        all_cats.extend(df.index.tolist())

    all_cats = list(pd.Index(all_cats).unique())

    if misc_threshold > 0 and misc_label not in all_cats:
        all_cats.append(misc_label)

    # ----------------------------------------------------
    # 2. SORT CATEGORIES BY GLOBAL FREQUENCY
    # ----------------------------------------------------
    global_totals = {cat: 0 for cat in all_cats}

    for df in abs_dict.values():
        for cat in df.index:
            global_totals[cat] += df.loc[cat].sum()

    sorted_cats = sorted(all_cats, key=lambda c: global_totals[c], reverse=True)

    # ----------------------------------------------------
    # 3. ASSIGN COLORS (consistent for both figures)
    # ----------------------------------------------------
    tol20 = [
        "#332288", "#88CCEE", "#44AA99", "#117733", "#999933",
        "#DDCC77", "#661100", "#CC6677", "#882255", "#AA4499",
        "#DDDDDD", "#000000", "#99DDFF", "#66CCEE", "#55AA77",
        "#BBCC33", "#EECC66", "#CC9966", "#AA8899", "#9999BB"
    ]

    shuffled = sorted_cats.copy()
    random.shuffle(shuffled)

    color_map = {cat: tol20[i % len(tol20)] for i, cat in enumerate(shuffled)}

    # ----------------------------------------------------
    # 4. HELPER FUNCTION TO PLOT AND SAVE FIGURE
    # ----------------------------------------------------
    def plot_figure(df_dict, value_mode="absolute", suffix=""):

        top_keys = list(df_dict.keys())
        n = len(top_keys)

        fig, axes = plt.subplots(1, n, figsize=(5*n, 6), sharey=True)
        if n == 1:
            axes = [axes]

        visible = set()

        for ax, top_key in zip(axes, top_keys):

            df = df_dict[top_key].copy()

            for cat in sorted_cats:
                if cat not in df.index:
                    df.loc[cat] = 0

            df = df.loc[sorted_cats]

            if value_mode == "percent":
                df = df.div(df.sum(axis=0), axis=1) * 100

            bottom = np.zeros(df.shape[1])
            x = np.arange(df.shape[1])

            for cat in df.index:
                values = df.loc[cat].values
                if values.sum() == 0:
                    continue

                visible.add(cat)

                ax.bar(
                    x, values,
                    bottom=bottom,
                    color=color_map[cat],
                    width=0.8
                )
                bottom += values

            ax.set_title(top_key.upper())
            ax.set_xticks(x)
            ax.set_xticklabels(df.columns, rotation=15)

            if ax == axes[0]:
                ax.set_ylabel("Count" if value_mode == "absolute" else "Percentage (%)")

        handles = []
        labels = []

        for cat in sorted_cats:
            if cat in visible:
                handles.append(plt.Line2D(
                    [0], [0],
                    marker='o', color='w',
                    markerfacecolor=color_map[cat],
                    markersize=10, label=cat
                ))
                labels.append(cat)

        if legend_loc == "right":
            fig.legend(handles, labels,
                       bbox_to_anchor=(1.02, 0.5),
                       loc="center left")
            plt.subplots_adjust(right=0.85)

        elif legend_loc == "bottom":
            fig.legend(handles, labels,
                       loc="lower center", ncol=4)
            plt.subplots_adjust(bottom=0.2)

        plt.tight_layout()

        # ----------------------------------------------------
        #  SAVE PNG + SVG (ADDED)
        # ----------------------------------------------------
        if save_as_png:
            # PNG
            fname_png = f"ytma_{save_as_png}_{suffix}.png"
            fig.savefig(os.path.join(output, fname_png), bbox_inches="tight")
            print(f"Saved: {fname_png}")

            # SVG
            fname_svg = f"ytma_{save_as_png}_{suffix}.svg"
            fig.savefig(os.path.join(output, fname_svg), bbox_inches="tight")
            print(f"Saved: {fname_svg}")

        plt.show()

    # ----------------------------------------------------
    # 5. PLOT BOTH FIGURES
    # ----------------------------------------------------
    plot_figure(abs_dict, value_mode="absolute", suffix="absolute")
    plot_figure(abs_dict, value_mode="percent",  suffix="percent")


In [ ]:
def createcrossgamecomparisondict(col, datadict, languages):
    tmp = {g: {} for g in datadict.keys()}
    results = {g: {} for g in datadict.keys()}
    for game, df_dict in datadict.items():
        print (game)
        print (df_dict.keys())
        tmp[game] = {l: df_dict[df_dict['videoSearchRegion'] == l] for l in languages}
        for l in languages:
            results[game][l] = tmp[game][l][col].value_counts(dropna=False)
            


#            results[game][lang] = {k: investigatefacet(df, col, k) for k in contentlabelpairs.keys()}
    return results

In [ ]:
col = 'Content Type (choose 0-1)'
comparisondict = createcrossgamecomparisondict(col, datadict, plot_order)
plot_nested_stacked_bars(comparisondict, misc_threshold=0.03, save_as_png=f"CategoryDistributionComparison_ZeldaSF6_{col}")

In [ ]:
col = 'Values (Subjective Evaluation) muliple possible'
comparisondict = createcrossgamecomparisondict(col, datadict, plot_order)
plot_nested_stacked_bars(comparisondict, misc_threshold=0.03, save_as_png=f"CategoryDistributionComparison_ZeldaSF6_{col}")

# compare two facets

! under construction !

## simple comparison

In [ ]:
def compare_column_distribution(dfs, col, labels=None, bins=20):
    """
    Compare the distribution of the same column in two (or more) DataFrames.
    
    Parameters:
    - dfs: list of DataFrames
    - col: column name to compare
    - labels: optional list of labels for each DataFrame
    - bins: number of bins for the histogram
    """
    if labels is None:
        labels = [f"DF {i+1}" for i in range(len(dfs))]

    plt.figure(figsize=(10, 5))

    for df, label in zip(dfs, labels):
        if col not in df.columns:
            print(f"⚠️ Column '{col}' not found in {label}")
            continue
        plt.hist(df[col].dropna(), bins=bins, alpha=0.6, label=label, density=True)

    plt.title(f"Distribution Comparison: '{col}'")
    if labels is None:
        plt.xlabel(col)
        plt.ylabel("Density")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# needs reworking

reaction_df = investigatefacet(data, 'Content Type (choose 0-1)', 'reaction')
nandf = data[data['Content Type (choose 0-1)'].isnull()]
target_column = 'Content Type (choose 0-1)'
#print(reaction_df.head())
for c in reaction_df.columns:
    if c not in exclude and c != target_column:
        compare_column_distribution([reaction_df, nandf], col=c, labels=["Reaction Videos", "Gameplay Videos"])

## normalized bar distribution

In [ ]:
def compare_column_distribution_bar(dfs, col, labels=None, bins=20):
    """
    Compare the distribution of categorical (string) values in the same column across multiple DataFrames.
    
    Parameters:
    - dfs: list of DataFrames
    - col: column name to compare
    - labels: optional list of labels for each DataFrame
    """
    if labels is None:
        labels = [f"DF {i+1}" for i in range(len(dfs))]

    plt.figure(figsize=(10, 5))

    # Define a color palette (use as many colors as the number of DataFrames)
    colors = plt.cm.get_cmap('tab10', len(dfs))

    # Create a list to store the width of each bar
    width = 0.8 / len(dfs)  # Adjust the width so that the bars are side-by-side

    for i, (df, label) in enumerate(zip(dfs, labels)):
        if col not in df.columns:
            print(f"⚠️ Column '{col}' not found in {label}")
            continue

        # Drop NaN values and ensure we're only dealing with valid data
        clean_data = df[col].dropna()

        # Check if the column has any data after dropping NaN
        if clean_data.empty:
            print(f"⚠️ No valid data in column '{col}' for {label}. Skipping plot for this DataFrame.")
            continue

        # Count the occurrences of each unique value in the column
        value_counts = clean_data.value_counts()

        # Normalize to get shares (so bars are comparable)
        value_shares = value_counts / value_counts.sum()


        # Get the positions for each category on the x-axis
        category_positions = range(len(value_shares))

        # Plot the bars for this DataFrame
        plt.bar(
            [pos + i * width for pos in category_positions],  # Adjust position to avoid overlap
            value_shares,  # Height of each bar
            width=width,  # Bar width
            label=label,  # Label for legend
            color=colors(i),  # Color for each
            alpha=0.7  # Transparency
                )
    plt.figure.text(0.04, 0.5, "Relative Frequency", va='center', rotation='vertical')

    # Add titles, labels, and grid
    plt.title(f"Categorical Distribution Comparison: '{col}'")
    plt.xlabel(col)
    plt.ylabel("Count")
    
    # Adjust the x-ticks so they are centered properly
    plt.xticks([pos + (len(dfs) - 1) * width / 2 for pos in category_positions], value_shares.index, rotation=45, ha='right', position=(0.1, 0))
    
    # Add legend and grid
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# needs reworking

reaction_df = investigatefacet(data, 'Content Type (choose 0-1)', 'reaction')
nandf = data[data['Content Type (choose 0-1)'].isnull()]

target_column = 'Content Type (choose 0-1)'
#print(reaction_df.head())
for c in focus_columns:
    if c not in exclude and c != target_column:
        compare_column_distribution_bar([reaction_df, nandf], col=c, labels=["Reaction Videos", "Gameplay Videos"])

## compare different languages

In [ ]:
def compare_column_distribution_bar_languagepanel_v2(game, df_dicts, col, labels=None, misc=False, pdf=None):
    def checkforcol(dict, cl):
        for k, d in dict.items():
            if cl not in d.columns:
                print(f"⚠️ Column '{cl}' not found in {k}")
                return False
        return True
    
    def determine_xaxis_labels(dicts, threshold):
        print('determining labels...')
        all_categories = set()
        
        for dict in dicts:
            #print(dict)
            total_items_dict = {k: dict[k][col].shape[0] for k in dict.keys()}
            counts_dict = {k: dict[k][col].value_counts(dropna=True) for k in dict.keys()}
            print('total items: ', total_items_dict)
            print('counts: ', counts_dict)
            for lang, counts in counts_dict.items(): #value_counts_dict.items():
                total_items = total_items_dict[lang]  # include NaNs in denominator
                print('labeling... determining shares for: ', counts)
                shares = counts / total_items  # normalize relative to all items
                # Split into main and small categories
                if threshold > 0:
                    main = shares[shares >= threshold]
                    small = shares[shares < threshold]
                    # Combine small categories into "misc"
                    if len(small) > 0:
                        print('labeling ... misc category added')
                        main.loc["misc"] = small.sum()
                    all_categories.update(main.keys())
                else:
                    all_categories.update(shares.keys())
        return all_categories

    all_cats = sorted(list(determine_xaxis_labels(df_dicts, 0.05)))# Sort the categories to ensure a consistent order
    category_positions = range(len(all_cats))  # Create positions based on the sorted categories

    print('categories available: ', all_cats)
    print('categories sorted: ', category_positions)

    """
    Compare the distribution of categorical (string) values in the same column across multiple DataFrames.
    
    Parameters:
    - df_dicts: list of dictionaries of DataFrames
    - col: column name to compare
    - labels: optional list of labels for each DataFrame
    """ 
    print(labels)       
    if labels is None:
        labels = [f"DF {i+1}" for i in range(len(df_dicts))]
    #figure, axes = plt.subplots(1, 4, figsize=(2*len(all_categories), 10), constrained_layout=True)
    figure, axes = plt.subplots(1, 4, constrained_layout=True)
    axes = axes.flatten()

        # Define a color palette (use as many colors as the number of DataFrames)
    colors = plt.cm.get_cmap('tab10', len(df_dicts))


    width = 0.8 / len(df_dicts)  # Adjust the width so that the bars are side-by-side
    #print(f"Bar width: {width}")
    #print('Setup complete for number of dicts: ', len(df_dicts))
    
    #print('checking base data')
    #for df_d in df_dicts:
        #print(df_d['ko'].head())



    # Loop through each dictionary (df_dict) and its corresponding label
    for i, (df_dict, label) in enumerate(zip(df_dicts, labels)):
        if col == 'relevanceLanguage' or checkforcol(df_dict, col) == False:
            print('Column does not exist or cannot be used in this context')
            continue
        #print(i, df_dict.keys(), label)
        print('label: ', label)
        #print('number of items per key\n')
        #print(k, [df_dict[k][col].shape[0] for k in df_dict.keys()])
        #print('value counts excluding nan per key\n')
        #print(k, [df_dict[k][col].value_counts(dropna=True) for k in df_dict.keys()])
        #print('overview concluded')
        # Total items per language including NaNs
        total_items_dict = {k: df_dict[k][col].shape[0] for k in df_dict.keys()}

        print('total items dict')
        print(total_items_dict)

        # Counts per language (excluding NaNs for categories)
        counts_dict = {k: df_dict[k][col].value_counts(dropna=True) for k in df_dict.keys()}

        print('counts dict')
        print(counts_dict)
        print('details')
        for l in counts_dict:
            print(l)
            print(counts_dict[l])
            for k, v in counts_dict[l].items():
                print(k, ': ', v, ' share: ', v/sum(counts_dict[l].values))
                #print()
        # Drop NaN values and ensure we're only dealing with valid data
        #clean_data_dict = {k: df_dict[k][col].dropna() for k in df_dict.keys()} 

        # Count the occurrences of each unique value in the column
        #print('Available languages:', clean_data_dict.keys())
        #value_counts_dict = {lang: clean_data_dict[lang].value_counts() for lang in clean_data_dict.keys()}

        # Normalize to get shares (so bars are compara
        #value_shares_dict = {lang: value_counts_dict[lang] / value_counts_dict[lang].sum() for lang in value_counts_dict.keys()}

        value_shares_dict = {}
        threshold = 0.05  # 5%

        n_labels = 0

        for lang, counts in counts_dict.items(): #value_counts_dict.items():
            total_items = total_items_dict[lang]  # include NaNs in denominator
            print('determining shares for: ', counts)
            shares = counts / total_items  # normalize relative to all items
            # Split into main and small categories
            if misc:
                print('misc category added')
                main = shares[shares >= threshold]
                small = shares[shares < threshold]
                print('main')
                print(main)
                print('small')
                print(small)

                # Combine small categories into "misc"
                if len(small) > 0:
                    main.loc["misc"] = small.sum()
                value_shares_dict[lang] = main
            else:
                value_shares_dict[lang] = shares
            
            if len(value_shares_dict[lang]) > n_labels:
                n_labels = len(value_shares_dict[lang])
            
            # Reorder by share (optional)
            #main = main.sort_values(ascending=False)
            print('results of determination: ', value_shares_dict[lang])
            print('results of determination (counts): ', counts)
        

        figure.set_size_inches(4*n_labels, 5) 

        nperlanguage = ''
        total_items = 0

        # Plot the data in a 2x2 grid for each dictionary (df)
#        for ax, (title, item) in zip(axes, value_shares_dict.items()):
        
        for ax, title in zip(axes, plot_order):
            try:
                item = value_shares_dict[title]
            except:
                print('no data for language: ', title)
                continue
            ax.set_title(title)
            #print(f"Plotting for {title}")
                
            # Align item data with the common categories
            aligned_item = item.reindex(all_cats, fill_value=0)  # Align to all_categories with 0 for missing categories

            # Total items for this dictionary/language (for legend)
            total_items += total_items_dict[title]
            nperlanguage += f'{title}:{total_items_dict[title]}, '
            
            # Plot the bars for this DataFrame
            ax.bar(
                [pos + i * width for pos in category_positions],  # Adjust position to avoid overlap
                aligned_item,  # Height of each bar (the relative frequency)
                width=width,  # Bar width
                label=f"{label} n= {str(total_items)} \n ({nperlanguage})", # (n={total_items})",
                color=colors(i),  # Color for each DataFrame
                alpha=0.7  # Transparency
                )
            
            ax.set_title(title)

            #ax.set_title(title + ' (n='+ str(value_counts_dict[title].sum())+')')
            ax.grid(True)
            
            # Adjust the x-ticks so they are centered properly
            ax.set_xticks([pos + (width * len(df_dicts) - 1) / 2 for pos in category_positions])
            ax.set_xticklabels(all_cats, rotation=45, ha='right', fontsize=12-(len(all_cats)/2))
            print('item index', item.index)

            for xtlabel in ax.get_xticklabels():
                xtlabel.set_transform(xtlabel.get_transform() + mtransforms.ScaledTranslation(15/len(all_cats)/72, 0, figure.dpi_scale_trans))

# Add titles, labels, and grid
    plt.suptitle(f"{game}: Categorical Distribution Comparison across Languages: \n '{col}'", fontsize=12)
    #plt.xlabel(col)
    plt.subplots_adjust(top=0.80)  # smaller = more space above the subplots

    figure.text(-0.04, 0.5, "Relative Frequency", va='center', rotation='vertical')
    plt.legend(
        loc='center left',
        bbox_to_anchor=(1.0, 0.5)
    )
    #plt.tight_layout(rect=[0, 0, 0.85, 1])
    #plt.legend(loc='best', title="Facets")
    plt.grid(axis='y', alpha=0.3)
    plt.subplots_adjust(hspace=1, wspace=0.3)

    # Adjust layout to prevent overlap
    #plt.tight_layout()
    # plt.show()
    # Save the current figure to the PDF
    if pdf:
        pdf.savefig(bbox_inches="tight")  # Save to the same PDF
        plt.close()  # Close the plot to prevent it from displaying in the notebook
        print(f"Plot for column '{col}' saved.")
    else:
        plt.show()


In [ ]:
def createreportofcategorydistribution(game, data, col, contentlabelpairs, misc_switch=False):
    resultsdict = {}
    for k, v in contentlabelpairs.items():
        print ('now processing', k, v)
        #print(data.head())
        tmp = investigatefacet(data, col, k)
        #print(tmp.head())
        print('unique languages in dict: ', tmp['relevanceLanguage'].unique())
        resultsdict[k] = {l: tmp.loc[tmp['relevanceLanguage']==l] for l in tmp['relevanceLanguage'].unique()}
        print('stats per language:')
        for l in resultsdict[k]:
            print(l, len(resultsdict[k][l]))
    output_pdf = os.path.join(gamedict[game], f'{game}_categorysystem_comparing{col}_misc{str(misc_switch)}_perLanguage_plots_v3.pdf')
    with PdfPages(output_pdf) as pdf:
        for c in focus_columns:
            if c not in exclude and c != col:
                compare_column_distribution_bar_languagepanel_v2(game, list(resultsdict.values()), col=c, labels=resultsdict.keys(), misc=misc_switch, pdf=pdf)
    print(f"All plots saved to {output_pdf}")
    #print(reaction_df.head())

In [ ]:
col = 'Content Type (choose 0-1)'
contentlabelp = {"reaction": "Reaction Videos",
                 "walkthrough": "Walkthrough Videos",
                 "nan": "Gameplay Videos",
                 "opinion/review": "Opinion or Review",
                 "experiment": "Experiment",
                 "creative": "Creative",
                 "tournament": "Tournament"}
#                 "gamecomparison": "Game comparison"}
for game, data in datadict.items():
    print(game)
    #print(data.head())
    createreportofcategorydistribution(game, data, col, contentlabelp, misc_switch=True)


In [ ]:
col = 'Values (Subjective Evaluation) muliple possible'
contentlabelp = {"informative": "Informative",
                 "entertainment": "Entertainment",
                 "community-building; entertainment": "Community-building and Entertainment",
                 "comedy": "Comedy",
                 "comedy; entertainment": "Comedy and Entertainment",
                 "community-building": "Community-building",
                 "promotional": "Promotional",
                 "entertainment; informative": "Entertainment and Informative"
                 }
for game, data in datadict.items():
    print(game)
    #print(data.head())
    createreportofcategorydistribution(game, data, col, contentlabelp, misc_switch=True)

In [ ]:
col = 'Content Focus (choose 0-2)'
contentlabelp = {
                 "equipment": "equipment",
                 "glitch": "glitch",
                 "crafting": "crafting",
                 "bossfight": "bossfight",
                 "story": "story",
                 "derivative": "derivative",
                 "character": "character",
                 "music/ost": "music/ost"}
for game, data in datadict.items():
    print(game)
    #print(data.head())
    createreportofcategorydistribution(game, data, col, contentlabelp, misc_switch=True)